# GCI World 2026 - Dual-Mechanism Strictly-Inductive Pipeline (one notebook, one CSV)

**Mechanism A (v17-style):** 3 stacking views (EXT, LOAN, PERSON) with out-of-fold disagreement logits `VD = logit_EXT - logit_LOAN ...`, unconstrained target encodings, LightGBM members + CatBoost product specialist.
**Mechanism B (v18-style):** 6 stacking views (adds EXTAGE, NOEXT, GROUPS), **monotone constraints** on target-encoded risk features (risk must not decrease as TE rises) and on the external scores (risk must not rise as the score rises), LightGBM members.

Both mechanisms are rebuilt from the raw files in this single notebook, so no earlier CSV, cache or prediction file is read.

## How your audit findings are handled
| Audit finding | What this notebook does |
|---|---|
| Matched-applicant label transfer (`partner_enc`, `PY_*`) | **Removed.** No feature ever carries another applicant's `TARGET`. Repeat-applicant matching is not used at all. |
| `pd.concat([train, test])` before feature fitting | **Removed.** Every cross-row statistic (category vocabularies, frequency counts, group means, fee-flag counts, key vocabularies, residual regressors, target encodings, stacking views) is fitted on the *training rows of the current fold only*, then applied to validation and test rows. |
| Residual regressors fitted on train+test | Fitted per fold on the fold's training rows; out-of-fold for those rows, fold-model average for validation/test. |
| Caches reused without provenance | **No cache of any kind.** Nothing is read except `train.csv`, `test.csv`, `sample_submission.csv`. |
| File provenance | SHA-256 of the three files is printed; fill `EXPECTED_SHA256` to enforce. |
| Test predictions | Average of the 5 fold models. Each fold model sees test features built only from that fold's training rows. |

## Routing policy (your old/new-customer idea)
The competition rules say that *"making manual decisions for part of the predictions based on rules or specific conditions derived from EDA also falls under Hand-Labeling."* A hand-picked split of the test set is therefore the riskiest part of the idea. So:
* The **default submission** sends every applicant through both mechanisms with one fixed, pre-declared rank blend (50/50).
* Routing is only an **experiment**: 10 pre-declared, row-intrinsic splits (constants, no cross-row statistics) are compared with the blend on out-of-fold predictions and against 30 random placebo splits. A routed strategy is used **only if** it beats the placebo 95th percentile, gains at least +0.0005 and wins in at least 4 of 5 folds. If nothing passes, the blend is submitted.
* "Old customer" in the sense of *repeat applicant* needs cross-row matching and label transfer, so it is deliberately **not** used.

Set `GCI_FAST=1` for a quick smoke run. Normal runs take a few hours on a 12-thread machine.

In [ ]:
# Cell 1: setup
import os, sys, time, glob, gc, hashlib, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingRegressor as HR, HistGradientBoostingClassifier as HC

FAST = os.environ.get('GCI_FAST', '0') == '1'
CB_GPU = os.environ.get('GCI_CB_GPU', '0') == '1'
NCPU = os.cpu_count() or 4
SEED = 42
np.random.seed(SEED)
RES_ITERS, VIEW_ITERS = (20, 20) if FAST else (100, 100)
rounds = lambda r: 25 if FAST else r
print('LightGBM', lgb.__version__, '| threads', NCPU, '| FAST =', FAST)

In [ ]:
# Cell 2: data (only the three official files) + provenance
def find_data():
    override = os.environ.get('GCI_DATA_DIR')
    if override:
        return [os.path.join(override, x) for x in ('train.csv', 'test.csv', 'sample_submission.csv')]
    roots = ('/content', '/content/drive/MyDrive', 'd:/gci/inpuy', 'd:/gci/input', os.getcwd())
    cands = []
    for r in roots:
        if os.path.isdir(r):
            for pat in ('train.csv', '*/train.csv', '*/*/train.csv'):
                cands += glob.glob(os.path.join(r, pat))
    for f in sorted(set(cands)):
        d = os.path.dirname(f)
        if all(os.path.exists(os.path.join(d, x)) for x in ('test.csv', 'sample_submission.csv')):
            head = open(f, 'rb').readline().decode('utf-8', 'ignore').strip().split(',')
            if len(head) == 34 and 'TARGET' in head:
                return [os.path.join(d, x) for x in ('train.csv', 'test.csv', 'sample_submission.csv')]
    raise FileNotFoundError('official competition files not found')

TRAIN_PATH, TEST_PATH, SAMPLE_PATH = find_data()
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
EXPECTED_SHA256 = {}          # e.g. {'train.csv': '...', 'test.csv': '...', 'sample_submission.csv': '...'}
for p in (TRAIN_PATH, TEST_PATH, SAMPLE_PATH):
    h = sha(p); n = os.path.basename(p)
    print(f'{n:22s} sha256 {h}')
    if n in EXPECTED_SHA256: assert h == EXPECTED_SHA256[n], f'hash mismatch for {n}'

df_train = pd.read_csv(TRAIN_PATH); df_test = pd.read_csv(TEST_PATH); sample = pd.read_csv(SAMPLE_PATH)
y = df_train['TARGET'].values.astype(np.int32)
df_train = df_train.drop(columns=['TARGET'])
ntr, nte = len(df_train), len(df_test)
assert (sample['SK_ID_CURR'].values == df_test['SK_ID_CURR'].values).all(), 'sample_submission order differs from test.csv'
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(ntr), y))
print(f'train {df_train.shape} test {df_test.shape} target rate {y.mean():.4f}')

In [ ]:
# Cell 3: row-wise features + statistics fitted on the FIT rows only
CATS = ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE',
        'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE']
COUNT_COLS = CATS + ['REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_ANNUITY', 'AMT_INCOME_TOTAL']
GROUP_KEYS = ['ORGANIZATION_TYPE', 'OCCUPATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'REGION_POPULATION_RELATIVE']
GROUP_VALS = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'DAYS_BIRTH', 'CREDIT_INC']
FEE_CONST = 129.5      # annuity fee offset used by the champion (the modal annuity gap in train is 130.5 = 29 x 4.5)
K_STEP = 0.0132        # financing-ratio lattice step used by the champion

def fit_stats(d):
    """Everything that looks across rows. d = the FIT rows only."""
    S = {'vocab': {c: sorted(d[c].dropna().unique()) for c in CATS}}
    S['counts'] = {c: d[c].value_counts() for c in COUNT_COLS}
    tmp = d.assign(CREDIT_INC=d.AMT_CREDIT / d.AMT_INCOME_TOTAL)
    S['gmean'] = {(k, v): tmp.groupby(k)[v].mean() for k in GROUP_KEYS for v in GROUP_VALS}
    q = d.AMT_ANNUITY / d.AMT_CREDIT; qb = (d.AMT_ANNUITY - FEE_CONST) / d.AMT_CREDIT
    S['q5'] = q.round(5).value_counts(); S['qb5'] = qb.round(5).value_counts()
    return S

nc = lambda d, dd: pd.DataFrame(dd, index=d.index)

def transform(d, S):
    """Apply fitted statistics to ANY frame (fit, validation or test). Uses no other rows."""
    d = d.reset_index(drop=True)
    X = d.drop(columns=['SK_ID_CURR']).copy()
    for c in CATS:
        codes = pd.Categorical(d[c], categories=S['vocab'][c]).codes.astype(float)
        codes[codes < 0] = np.nan
        X[c] = codes
    raw = X.copy(); raw['DAYS_EMPLOYED'] = raw.DAYS_EMPLOYED.where(raw.DAYS_EMPLOYED != 365243)

    # clean
    dd = {'DE_SENT': (d.DAYS_EMPLOYED == 365243).astype(int), 'DAYS_EMPLOYED_CLEAN': d.DAYS_EMPLOYED.where(d.DAYS_EMPLOYED != 365243),
          'ORG_XNA': (d.ORGANIZATION_TYPE == 'XNA').astype(int), 'CAR_AGE_OUT': (d.OWN_CAR_AGE >= 60).astype(int), 'MISS_CNT': d.isna().sum(axis=1)}
    for c in ['EXT_SOURCE_1', 'EXT_SOURCE_3', 'OWN_CAR_AGE', 'AMT_REQ_CREDIT_BUREAU_YEAR']: dd[c + '_NA'] = d[c].isna().astype(int)
    Xc = pd.concat([X, nc(d, dd)], axis=1)
    # financial ratios
    inc, cr, an, gp = d.AMT_INCOME_TOTAL, d.AMT_CREDIT, d.AMT_ANNUITY, d.AMT_GOODS_PRICE
    dd = {'CREDIT_INC': cr / inc, 'ANN_INC': an / inc, 'ANN_CREDIT': an / cr, 'TERM': cr / an, 'GOODS_CREDIT': gp / cr, 'CREDIT_MINUS_GOODS': cr - gp,
          'GOODS_INC': gp / inc, 'INC_PER_FAM': inc / d.CNT_FAM_MEMBERS, 'CREDIT_PER_FAM': cr / d.CNT_FAM_MEMBERS, 'ANN_PER_FAM': an / d.CNT_FAM_MEMBERS}
    for c in ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']: dd['LOG_' + c] = np.log1p(d[c])
    Xc = pd.concat([Xc, nc(d, dd)], axis=1)
    # age / tenure
    age = -d.DAYS_BIRTH; de = d.DAYS_EMPLOYED.where(d.DAYS_EMPLOYED != 365243)
    dd = {'AGE': age / 365.25, 'EMP_AGE': de / d.DAYS_BIRTH, 'EMP_YEARS': -de / 365.25, 'REG_AGE': d.DAYS_REGISTRATION / d.DAYS_BIRTH,
          'ID_AGE': d.DAYS_ID_PUBLISH / d.DAYS_BIRTH, 'PHONE_AGE': d.DAYS_LAST_PHONE_CHANGE / d.DAYS_BIRTH, 'CAR_AGE_RATIO': d.OWN_CAR_AGE / (age / 365.25),
          'CREDIT_PER_AGE': d.AMT_CREDIT / (age / 365.25), 'INC_PER_AGE': d.AMT_INCOME_TOTAL / (age / 365.25)}
    Xc = pd.concat([Xc, nc(d, dd)], axis=1)
    # bureau enquiries
    b = [c for c in d.columns if c.startswith('AMT_REQ')]
    dd = {'BUR_SUM': d[b].sum(axis=1, min_count=1), 'BUR_RECENT': d[['AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_MON']].sum(axis=1, min_count=1),
          'BUR_MON_YEAR': d.AMT_REQ_CREDIT_BUREAU_MON / (d.AMT_REQ_CREDIT_BUREAU_YEAR + 1.0)}
    Xc = pd.concat([Xc, nc(d, dd)], axis=1)
    # external scores
    e = d[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']]
    dd = {'EXT_MEAN': e.mean(axis=1), 'EXT_MIN': e.min(axis=1), 'EXT_MAX': e.max(axis=1), 'EXT_STD': e.std(axis=1), 'EXT_RANGE': e.max(axis=1) - e.min(axis=1),
          'EXT_NOBS': e.notna().sum(axis=1), 'EXT_PROD': e.prod(axis=1), 'EXT_W': (d.EXT_SOURCE_1 * 2 + d.EXT_SOURCE_2 * 3 + d.EXT_SOURCE_3 * 4) / 9.0,
          'E12': d.EXT_SOURCE_1 * d.EXT_SOURCE_2, 'E13': d.EXT_SOURCE_1 * d.EXT_SOURCE_3, 'E23': d.EXT_SOURCE_2 * d.EXT_SOURCE_3,
          'D12': d.EXT_SOURCE_1 - d.EXT_SOURCE_2, 'D13': d.EXT_SOURCE_1 - d.EXT_SOURCE_3, 'D23': d.EXT_SOURCE_2 - d.EXT_SOURCE_3}
    Xa = pd.concat([Xc, nc(d, dd)], axis=1)
    # frequency counts and group-mean differences, from FIT-row statistics
    dd = {}
    for c in COUNT_COLS:
        m = d[c].map(S['counts'][c]); dd['CNT_' + c] = m.where(m.notna() | d[c].isna(), 0.0)
    Xa = pd.concat([Xa, nc(d, dd)], axis=1)
    tmp = d.assign(CREDIT_INC=d.AMT_CREDIT / d.AMT_INCOME_TOTAL); dd = {}
    for k in GROUP_KEYS:
        for v in GROUP_VALS: dd[f'G_{k[:6]}_{v[-6:]}_DIFF'] = tmp[v] - tmp[k].map(S['gmean'][(k, v)])
    Xa = pd.concat([Xa, nc(d, dd)], axis=1)
    # financing-ratio lattice index
    cash = d.NAME_CONTRACT_TYPE == 'Cash loans'
    kf = (cr / gp - 1.0) / K_STEP; k = kf.round(); ok = cash & (cr > gp) & ((kf - k).abs() < 0.1)
    K = pd.Series(np.where(ok, k, np.where(cash & (cr == gp), 0.0, np.nan)), index=d.index)
    kd = nc(d, {'K_DEC': K, 'CG_RATIO': cr / gp})
    Xck, Xak = pd.concat([Xc, kd], axis=1), pd.concat([Xa, kd], axis=1)
    # decoded product keys (fee flag from FIT-row counts)
    q = an / cr; qb = (an - FEE_CONST) / cr; q5, qb5 = q.round(5), qb.round(5)
    cnt = lambda s, c: s.map(c).fillna(0).values
    FEE = ((cnt(qb5, S['qb5']) > cnt(q5, S['q5'])) & cash.values).astype(int)
    Q0 = pd.Series(np.where(FEE, qb, q), index=d.index)
    CT = cr / an; CT1, CT2, CT3 = CT.round(1).astype(str), CT.round(2).astype(str), CT.round(3).astype(str)
    KS = {'Q3': q.round(3).astype(str), 'Q4': q.round(4).astype(str), 'Q5': q.round(5).astype(str), 'P3': Q0.round(3).astype(str), 'P4': Q0.round(4).astype(str),
          'P5': Q0.round(5).astype(str), 'PF4': Q0.round(4).astype(str) + '|' + pd.Series(FEE, index=d.index).astype(str),
          'CT1': CT1, 'CT2': CT2, 'CT3': CT3, 'CT1_s10': CT1, 'CT1_s200': CT1, 'CT1_GP': CT1 + '|' + gp.astype(str), 'CT1_CR': CT1 + '|' + cr.astype(str),
          'CT1_AN': CT1 + '|' + an.astype(str), 'GP': gp.astype(str), 'CR': cr.astype(str), 'AN': an.astype(str)}
    return {'d': d, 'raw': raw, 'Xc': Xc, 'Xa': Xa, 'Xck': Xck, 'Xak': Xak, 'KS': KS, 'Q0': Q0, 'FEE': FEE}

KEYS = ['Q3', 'Q4', 'Q5', 'P3', 'P4', 'P5', 'PF4', 'CT1', 'CT2', 'CT3', 'CT1_s10', 'CT1_s200', 'CT1_GP', 'CT1_CR', 'CT1_AN', 'GP', 'CR', 'AN']
M0 = {'Q3': 100., 'Q4': 30., 'Q5': 30., 'P3': 100., 'P4': 30., 'P5': 30., 'PF4': 30., 'CT1': 50., 'CT2': 50., 'CT3': 50., 'CT1_s10': 10., 'CT1_s200': 200.,
      'CT1_GP': 50., 'CT1_CR': 50., 'CT1_AN': 50., 'GP': 50., 'CR': 50., 'AN': 50.}
print('feature definitions loaded')

In [ ]:
# Cell 4: fold-local residuals, target encodings, stacking views, one-hot, CatBoost frame
def resid_targets(raw):
    demo = [c for c in raw.columns if c.startswith('EXT_SOURCE') or c.startswith('AMT_')]
    return {'EXT1': (raw.EXT_SOURCE_1, ['EXT_SOURCE_1']), 'EXT2': (raw.EXT_SOURCE_2, ['EXT_SOURCE_2']), 'EXT3': (raw.EXT_SOURCE_3, ['EXT_SOURCE_3']),
            'LOGINC': (np.log(raw.AMT_INCOME_TOTAL), ['AMT_INCOME_TOTAL']), 'LOGCR': (np.log(raw.AMT_CREDIT), ['AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE']),
            'EMP': (raw.DAYS_EMPLOYED, ['DAYS_EMPLOYED']), 'REG': (raw.DAYS_REGISTRATION, ['DAYS_REGISTRATION']), 'AGE': (raw.DAYS_BIRTH, ['DAYS_BIRTH']),
            'IDP': (raw.DAYS_ID_PUBLISH, ['DAYS_ID_PUBLISH']), 'PHONE': (raw.DAYS_LAST_PHONE_CHANGE, ['DAYS_LAST_PHONE_CHANGE']),
            'REGPOP': (raw.REGION_POPULATION_RELATIVE, ['REGION_POPULATION_RELATIVE']), 'FAM': (raw.CNT_FAM_MEMBERS, ['CNT_FAM_MEMBERS']),
            'CAR': (raw.OWN_CAR_AGE, ['OWN_CAR_AGE']), 'EXT1D': (raw.EXT_SOURCE_1, demo), 'EXT2D': (raw.EXT_SOURCE_2, demo), 'EXT3D': (raw.EXT_SOURCE_3, demo)}

def resid_fit(raw_fit):
    spec = resid_targets(raw_fit); models, out = {}, {}
    for nm, (yt, drop) in spec.items():
        Xp = raw_fit.drop(columns=drop); obs = yt.notna().values; pred = np.zeros(len(Xp)); ms = []
        for tri, tei in KFold(5, shuffle=True, random_state=0).split(Xp):
            m = HR(max_iter=RES_ITERS, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=100, random_state=0).fit(Xp.iloc[tri][obs[tri]], yt.iloc[tri][obs[tri]])
            pred[tei] = m.predict(Xp.iloc[tei]); ms.append(m)
        models[nm] = (ms, list(Xp.columns))
        out['RP_' + nm] = pred; out['RR_' + nm] = np.where(obs, yt.values - pred, np.nan)
    return pd.DataFrame(out), models

def resid_apply(raw_a, models):
    spec = resid_targets(raw_a); out = {}
    for nm, (ms, cols) in models.items():
        yt = spec[nm][0]; pred = np.mean([m.predict(raw_a[cols]) for m in ms], axis=0)
        out['RP_' + nm] = pred; out['RR_' + nm] = np.where(yt.notna().values, yt.values - pred, np.nan)
    return pd.DataFrame(out)

def key_codes(KS_fit, KS_apps):
    cf, ca, uniq = {}, {nm: {} for nm in KS_apps}, {}
    for k in KEYS:
        idx = pd.Index(KS_fit[k].unique()); cf[k] = idx.get_indexer(KS_fit[k]); uniq[k] = len(idx)
        for nm, KSa in KS_apps.items(): ca[nm][k] = idx.get_indexer(KSa[k])
    return cf, ca, uniq

def enc_codes(cf, yfit, fit_pos, qcodes, uniq):
    pr = float(yfit[fit_pos].mean()); yf = yfit[fit_pos].astype(float); out = {}
    for k in KEYS:
        c = cf[k][fit_pos]; cnt = np.bincount(c, minlength=uniq[k]).astype(float); sm = np.bincount(c, weights=yf, minlength=uniq[k])
        cq = qcodes[k]; ok = cq >= 0
        te = np.full(len(cq), pr); te[ok] = (sm[cq[ok]] + M0[k] * pr) / (cnt[cq[ok]] + M0[k])
        cn = np.zeros(len(cq)); cn[ok] = cnt[cq[ok]]
        out['TE_' + k] = te; out['CNT_' + k] = cn
    return pd.DataFrame(out)

lg = lambda p: np.log(np.clip(p, 1e-4, 1.0 - 1e-4) / (1.0 - np.clip(p, 1e-4, 1.0 - 1e-4)))
VIEW_A_COLS = ['VIEW_EXT', 'VIEW_LOAN', 'VIEW_PERSON', 'VD_EL', 'VD_EP', 'VD_LP']                 # mechanism A: 3 views
VIEW_B_COLS = VIEW_A_COLS + ['VIEW_EXTAGE', 'VIEW_NOEXT', 'VIEW_GROUPS']                           # mechanism B: 6 views

def view_defs(cols):
    ext = [c for c in cols if 'EXT' in c or c in ('E12', 'E13', 'E23', 'D12', 'D13', 'D23')]
    loan = [c for c in cols if c.startswith(('AMT_', 'LOG_', 'CREDIT_', 'ANN_', 'GOODS_', 'TERM', 'Q0num')) or c in ('CREDIT_MINUS_GOODS', 'RP_LOGINC', 'RR_LOGINC', 'RP_LOGCR', 'RR_LOGCR')]
    person = [c for c in cols if c not in set(ext) | set(loan) and not c.startswith(('G_', 'CNT_'))]
    agec = [c for c in cols if any(t in c for t in ('AGE', 'BIRTH', 'EMPLOY', 'EMP_', 'ID_', 'REG_', 'PHONE', 'DAYS'))]
    grpc = [c for c in cols if c.startswith(('G_', 'CNT_')) or c in CATS]
    return {'EXT': ext, 'LOAN': loan, 'PERSON': person, 'EXTAGE': sorted(set(ext + agec)), 'NOEXT': sorted(set(loan + person)), 'GROUPS': sorted(set(grpc + loan[:6]))}

def views_fit_apply(Xs_fit, y_fit, Xs_apps):
    views = view_defs(list(Xs_fit.columns)); vfit, vapp = {}, {nm: {} for nm in Xs_apps}
    mk = lambda: HC(max_iter=VIEW_ITERS, learning_rate=0.08, max_leaf_nodes=15, min_samples_leaf=150, l2_regularization=20, random_state=0)
    for vn, vc in views.items():
        inner = np.zeros(len(Xs_fit)); pa = {nm: [] for nm in Xs_apps}
        for ia, ib in StratifiedKFold(3, shuffle=True, random_state=11).split(np.zeros(len(Xs_fit)), y_fit):
            m = mk().fit(Xs_fit.iloc[ia][vc], y_fit[ia]); inner[ib] = m.predict_proba(Xs_fit.iloc[ib][vc])[:, 1]
            for nm, Xa_ in Xs_apps.items(): pa[nm].append(m.predict_proba(Xa_[vc])[:, 1])
        vfit['VIEW_' + vn] = lg(inner)
        for nm in Xs_apps: vapp[nm]['VIEW_' + vn] = lg(np.mean(pa[nm], 0))
    def vf(dct):
        df = pd.DataFrame(dct); df['VD_EL'] = df.VIEW_EXT - df.VIEW_LOAN; df['VD_EP'] = df.VIEW_EXT - df.VIEW_PERSON; df['VD_LP'] = df.VIEW_LOAN - df.VIEW_PERSON
        return df
    return vf(vfit), {nm: vf(v) for nm, v in vapp.items()}

def onehot_fit_apply(Xfit, Xapps):
    vocab = {c: sorted(Xfit[c].dropna().unique()) for c in CATS}
    def tr(X):
        cols = {}
        for ci, c in enumerate(CATS):
            for v in vocab[c]: cols[f'OH_{ci}_{int(v)}'] = (X[c].values == v).astype(np.float32)
            cols[f'OH_{ci}_nan'] = X[c].isna().values.astype(np.float32)
        return pd.concat([X.drop(columns=CATS), pd.DataFrame(cols, index=X.index)], axis=1)
    return tr(Xfit), {nm: tr(X) for nm, X in Xapps.items()}

PROD_KEYS = ['Q3', 'Q4', 'Q5', 'P3', 'P4', 'P5', 'PF4', 'CT1', 'CT2', 'CT3', 'CT1_GP', 'CT1_CR', 'CT1_AN']
def cb_frame(T):
    d = T['d']; num = T['Xa'].drop(columns=CATS).reset_index(drop=True)
    cat = pd.DataFrame({c: d[c].astype('string').fillna('__MISSING__').astype(str) for c in CATS})
    prod = pd.DataFrame({'PROD_' + k: T['KS'][k].astype('string').fillna('__MISSING__').astype(str) for k in PROD_KEYS})
    prod['PROD_K_DEC'] = T['Xak']['K_DEC'].round().astype('Int64').astype('string').fillna('__MISSING__').astype(str)
    prod['PROD_FEE'] = pd.Series(T['FEE']).astype(str)
    return pd.concat([num, cat, prod], axis=1)
CB_CATS = CATS + ['PROD_' + k for k in PROD_KEYS] + ['PROD_K_DEC', 'PROD_FEE']

def build_fold(df_fit, y_fit, apps):
    """Fit every cross-row statistic on df_fit ONLY and transform fit + apply frames. Returns all model inputs."""
    t0 = time.time()
    df_fit = df_fit.reset_index(drop=True); apps = {k: v.reset_index(drop=True) for k, v in apps.items()}
    S = fit_stats(df_fit)
    TF = transform(df_fit, S); TA = {k: transform(v, S) for k, v in apps.items()}
    rf, models = resid_fit(TF['raw']); RA = {k: resid_apply(TA[k]['raw'], models) for k in apps}
    withres = lambda T, R: {n: pd.concat([T[n], R], axis=1) for n in ('Xc', 'Xa', 'Xck', 'Xak')}
    FF = withres(TF, rf); FA = {k: withres(TA[k], RA[k]) for k in apps}
    # target encodings: out-of-fold for fit rows, fit-row statistics for the others
    cf, ca, uniq = key_codes(TF['KS'], {k: TA[k]['KS'] for k in apps})
    nfit = len(df_fit); te_fit = None
    for ia, ib in StratifiedKFold(5, shuffle=True, random_state=7).split(np.zeros(nfit), y_fit):
        part = enc_codes(cf, y_fit, ia, {k: cf[k][ib] for k in KEYS}, uniq)
        if te_fit is None: te_fit = pd.DataFrame(np.zeros((nfit, part.shape[1])), columns=part.columns)
        te_fit.iloc[ib] = part.values
    te_app = {k: enc_codes(cf, y_fit, np.arange(nfit), ca[k], uniq) for k in apps}
    # stacking views
    Xs_fit = FF['Xa'].copy(); Xs_fit['Q0num'] = TF['Q0'].values
    Xs_app = {}
    for k in apps:
        Xs_app[k] = FA[k]['Xa'].copy(); Xs_app[k]['Q0num'] = TA[k]['Q0'].values
    vfit, vapp = views_fit_apply(Xs_fit, y_fit, Xs_app)
    # member input frames
    M = {}
    co_f, co_a = onehot_fit_apply(FF['Xc'], {k: FA[k]['Xc'] for k in apps})
    ao_f, ao_a = onehot_fit_apply(FF['Xa'], {k: FA[k]['Xa'] for k in apps})
    ak_f, ak_a = onehot_fit_apply(FF['Xak'], {k: FA[k]['Xak'] for k in apps})
    M['Xco'] = {'fit': co_f, **co_a}; M['Xao'] = {'fit': ao_f, **ao_a}; M['Xaok'] = {'fit': ak_f, **ak_a}
    M['Xck'] = {'fit': FF['Xck'], **{k: FA[k]['Xck'] for k in apps}}
    cb = {'fit': cb_frame({**TF, 'Xa': TF['Xa'], 'Xak': TF['Xak']}), **{k: cb_frame(TA[k]) for k in apps}}
    print(f'  [fold features] fit rows {nfit} | built in {time.time()-t0:.0f}s', flush=True)
    return {'M': M, 'te': {'fit': te_fit, **te_app}, 'view': {'fit': vfit, **vapp}, 'cb': cb}
print('fold builder ready')

In [ ]:
# Cell 5: audit checkpoint - training features must not depend on test rows
def check_no_test_dependence(n_fit=6000):
    rng = np.random.default_rng(0); idx = rng.choice(ntr, min(n_fit, ntr), replace=False)
    dfit, yfit = df_train.iloc[idx], y[idx]
    t1 = df_test.iloc[:1500]
    t3 = t1.copy(); t3['AMT_CREDIT'] = t3['AMT_CREDIT'] * 1.3; t3['EXT_SOURCE_2'] = t3['EXT_SOURCE_2'].fillna(0.5) ** 2
    t3['ORGANIZATION_TYPE'] = 'Brand New Org'; t3 = t3.sample(frac=1.0, random_state=1)
    B1 = build_fold(dfit, yfit, {'test': t1}); B3 = build_fold(dfit, yfit, {'test': t3})
    for key in ('Xco', 'Xao', 'Xaok', 'Xck'):
        pd.testing.assert_frame_equal(B1['M'][key]['fit'], B3['M'][key]['fit'], rtol=1e-9, atol=1e-12)
    pd.testing.assert_frame_equal(B1['te']['fit'], B3['te']['fit'], rtol=1e-9, atol=1e-12)
    pd.testing.assert_frame_equal(B1['view']['fit'], B3['view']['fit'], rtol=1e-9, atol=1e-12)
    pd.testing.assert_frame_equal(B1['cb']['fit'], B3['cb']['fit'])
    print('[PASS] training feature matrices are identical when the test rows are changed, reordered and given unseen categories')
check_no_test_dependence()

In [ ]:
# Cell 6: model members. A = v17-style (3 views, unconstrained TE) ; B = v18-style (6 views, monotone TE)
BASE_LGB = dict(objective='binary', force_col_wise=True, bagging_freq=1, verbose=-1, num_threads=NCPU, seed=SEED, feature_pre_filter=False)
P_SHALLOW = dict(learning_rate=0.05, num_leaves=7, max_depth=3, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20)
P_REG     = dict(learning_rate=0.03, num_leaves=11, min_child_samples=300, feature_fraction=0.4, bagging_fraction=0.8, lambda_l2=40)
P_ALLFAM  = dict(learning_rate=0.04, num_leaves=15, min_child_samples=150, feature_fraction=0.3, bagging_fraction=0.8, lambda_l2=20)
P_MONO    = dict(learning_rate=0.02, num_leaves=15, min_child_samples=150, feature_fraction=0.5, bagging_fraction=0.8, lambda_l2=20)
ET = dict(extra_trees=True)
# name: (frame key, params, rounds, mechanism, monotone TE/EXT constraints?, blend weight inside the mechanism)
MEMBERS = {
    'A_qf_shallow':   ('Xco',  P_SHALLOW,       1300, 'A', False, 0.20),
    'A_qf_reg':       ('Xco',  P_REG,           1939, 'A', False, 0.10),
    'A_allfam_oh':    ('Xao',  P_ALLFAM,         881, 'A', False, 0.35),
    'A_allfam_oh_et': ('Xao',  {**P_ALLFAM, **ET}, 881, 'A', False, 0.35),
    'B_mono':         ('Xck',  P_MONO,          1571, 'B', True,  0.25),
    'B_allfam_oh':    ('Xaok', P_ALLFAM,         881, 'B', True,  0.30),
    'B_allfam_oh_et': ('Xaok', {**P_ALLFAM, **ET}, 881, 'B', True,  0.30),
    'B_qf_reg_et':    ('Xco',  {**P_REG, **ET}, 1939, 'B', True,  0.15)}
VIEWCOLS = {'A': VIEW_A_COLS, 'B': VIEW_B_COLS}
rk = lambda v: (rankdata(v) - 1.0) / (len(v) - 1.0)

def mono_vector(cols, mono_te):
    return [-1 if c in ('EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3') else (1 if (mono_te and c.startswith('TE_')) else 0) for c in cols]

oof = {m: np.zeros(ntr) for m in MEMBERS}; tst = {m: np.zeros(nte) for m in MEMBERS}
cb_oof = np.zeros(ntr); cb_tst = np.zeros(nte)
t_all = time.time()
for k, (a, b) in enumerate(folds, 1):
    print(f'=== outer fold {k}/5 ===', flush=True)
    FB = build_fold(df_train.iloc[a], y[a], {'val': df_train.iloc[b], 'test': df_test})
    for name, (fk, params, R, mech, mono, w) in MEMBERS.items():
        vc = VIEWCOLS[mech]
        fa = pd.concat([FB['M'][fk]['fit'].reset_index(drop=True), FB['te']['fit'], FB['view']['fit'][vc]], axis=1)
        fv = pd.concat([FB['M'][fk]['val'].reset_index(drop=True), FB['te']['val'], FB['view']['val'][vc]], axis=1)
        ft = pd.concat([FB['M'][fk]['test'].reset_index(drop=True), FB['te']['test'], FB['view']['test'][vc]], axis=1)
        p = {**BASE_LGB, **params}
        if mech == 'B': p['monotone_constraints'] = mono_vector(list(fa.columns), mono)
        m = lgb.train(p, lgb.Dataset(fa, y[a]), rounds(R))
        oof[name][b] = m.predict(fv); tst[name] += m.predict(ft) / 5.0
        print(f'  fold {k} {name:15s} AUC {roc_auc_score(y[b], oof[name][b]):.5f}', flush=True)
    cbp = dict(iterations=30 if FAST else 900, depth=7, learning_rate=0.05, loss_function='Logloss', eval_metric='AUC', l2_leaf_reg=8, random_seed=SEED + k,
               thread_count=NCPU, verbose=False, allow_writing_files=False)
    if CB_GPU: cbp['task_type'] = 'GPU'
    cm = CatBoostClassifier(**cbp); cm.fit(FB['cb']['fit'], y[a], cat_features=CB_CATS)
    cb_oof[b] = cm.predict_proba(FB['cb']['val'])[:, 1]; cb_tst += cm.predict_proba(FB['cb']['test'])[:, 1] / 5.0
    print(f'  fold {k} CatBoost        AUC {roc_auc_score(y[b], cb_oof[b]):.5f}', flush=True)
    del FB; gc.collect()
print(f'training finished in {(time.time()-t_all)/60:.1f} min')

In [ ]:
# Cell 7: assemble the two mechanisms (fixed pre-declared weights)
def mech_blend(mech, arrays):
    names = [n for n in MEMBERS if MEMBERS[n][3] == mech]
    s = sum(MEMBERS[n][5] * rk(arrays[n]) for n in names)
    return s
mA_oof = rk(0.9 * rk(mech_blend('A', oof)) + 0.1 * rk(cb_oof)); mA_tst = rk(0.9 * rk(mech_blend('A', tst)) + 0.1 * rk(cb_tst))
mB_oof = rk(mech_blend('B', oof));                                mB_tst = rk(mech_blend('B', tst))
for nm, arr in [(n, oof[n]) for n in MEMBERS] + [('CatBoost', cb_oof), ('MECHANISM A (v17-style)', mA_oof), ('MECHANISM B (v18-style)', mB_oof)]:
    print(f'{nm:26s} OOF AUC {roc_auc_score(y, arr):.5f}')
W_A = 0.5   # pre-declared, applies to every applicant
blend_oof = rk(W_A * mA_oof + (1 - W_A) * mB_oof); blend_tst = rk(W_A * mA_tst + (1 - W_A) * mB_tst)
fold_auc = lambda s: [roc_auc_score(y[b], s[b]) for _, b in folds]
print(f'\nUNIFORM BLEND (default)  OOF AUC {roc_auc_score(y, blend_oof):.5f} | folds {np.round(fold_auc(blend_oof), 5).tolist()}')
print(f'Spearman(A,B) on OOF = {np.corrcoef(rankdata(mA_oof), rankdata(mB_oof))[0, 1]:.5f}')

In [ ]:
# Cell 8: routing experiment (gated). Splits are row-intrinsic constants: no cross-row statistics, no labels, no matching of applicants.
def seg_masks(d):
    age = -d.DAYS_BIRTH / 365.25; emp = d.DAYS_EMPLOYED.where(d.DAYS_EMPLOYED != 365243); cash = d.NAME_CONTRACT_TYPE == 'Cash loans'
    M = {'long_tenure_ge5y': emp <= -1825, 'age_ge_45': age >= 45, 'old_registration': d.DAYS_REGISTRATION <= -5000, 'old_phone_change': d.DAYS_LAST_PHONE_CHANGE <= -1000,
         'owns_car': d.OWN_CAR_AGE.notna(), 'pensioner': d.NAME_INCOME_TYPE == 'Pensioner', 'cash_loan': cash,
         'ext_all_present': d[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].notna().all(axis=1),
         'insured_premium': cash & (d.AMT_CREDIT > d.AMT_GOODS_PRICE * 1.002), 'high_leverage': d.AMT_CREDIT / d.AMT_INCOME_TOTAL > 3.5}
    return {k: v.fillna(False).values.astype(bool) for k, v in M.items()}

SEG_TR, SEG_TE = seg_masks(df_train), seg_masks(df_test)
base_auc = roc_auc_score(y, blend_oof); base_folds = np.array(fold_auc(blend_oof))
def routed(mask, rA, rB, a_on_mask): return np.where(mask, rA, rB) if a_on_mask else np.where(mask, rB, rA)
rows = []
for sname, mask in SEG_TR.items():
    for a_on in (True, False):
        s = routed(mask, mA_oof, mB_oof, a_on)
        rows.append((sname, 'A' if a_on else 'B', int(mask.sum()), roc_auc_score(y, s) - base_auc, int((np.array(fold_auc(s)) > base_folds).sum())))
R = pd.DataFrame(rows, columns=['segment', 'mechanism_on_segment', 'rows_in_segment', 'delta_vs_blend', 'folds_better_of_5'])
rng = np.random.default_rng(2026); plc = []
for i in range(30):
    mask = rng.random(ntr) < rng.choice([0.2, 0.35, 0.5, 0.65, 0.8])
    for a_on in (True, False): plc.append(roc_auc_score(y, routed(mask, mA_oof, mB_oof, a_on)) - base_auc)
thr = max(0.0005, float(np.quantile(plc, 0.95)))
print(f'placebo routing deltas: median {np.median(plc):+.5f} | 95th percentile {np.quantile(plc, 0.95):+.5f} | gate threshold {thr:+.5f}')
print(R.sort_values('delta_vs_blend', ascending=False).head(10).to_string(index=False))
passing = R[(R.delta_vs_blend >= thr) & (R.folds_better_of_5 >= 4)].sort_values('delta_vs_blend', ascending=False)
if len(passing):
    best = passing.iloc[0]; print('\nROUTING PASSES THE GATE ->', best.segment, 'with mechanism', best.mechanism_on_segment, 'on the segment')
    final = routed(SEG_TE[best.segment], mA_tst, mB_tst, best.mechanism_on_segment == 'A'); STRATEGY = f'routed:{best.segment}:{best.mechanism_on_segment}'
else:
    print('\nNo routed strategy passes the gate -> submitting the uniform blend (every applicant, same fixed weights).')
    final = blend_tst; STRATEGY = 'uniform_blend_50_50'
print('STRATEGY =', STRATEGY)

In [ ]:
# Cell 9: write ONE csv + integrity checks
final = (rankdata(final) - 1.0) / (len(final) - 1.0)
sub = pd.DataFrame({'SK_ID_CURR': df_test['SK_ID_CURR'].values, 'TARGET': final})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert sub['SK_ID_CURR'].is_unique and (sub['SK_ID_CURR'].values == sample['SK_ID_CURR'].values).all()
assert sub['TARGET'].notna().all() and np.isfinite(sub['TARGET']).all() and sub['TARGET'].between(0, 1).all()
OUT = 'GCI_dual_mechanism_submission.csv'
sub.to_csv(OUT, index=False)
print(f'[+] wrote {OUT} | rows {len(sub)} | strategy {STRATEGY} | sha256 {sha(OUT)}')
print('Rule checklist: official files only | no cache | no label transfer between applicants | no train+test pooling | no manual test-prediction changes | one CSV')